# Goodreads Books Dataset: Data Preparation & Cleaning Pipeline

**Project Role:** Senior Data Engineer  
**Stage:** Phase 1 – Data Ingestion, Inspection, Validation, and Preprocessing  
**Target Deliverables:** `books_cleaned.csv`, `data_cleaning.ipynb`, `requirements.txt`

---

### Project Scope & Engineering Boundaries
In this collaborative two-member project, this notebook encompasses the complete **Data Preparation and Data Cleaning** lifecycle. The primary engineering objective is to transform raw, unvalidated bibliographic data into an audited, standardized, type-safe, and validated dataset ready for downstream analytics and machine learning.

**Explicit Scope Constraints:**
* Strictly focused on data engineering, parsing anomaly resolution, schema auditing, missing value treatment, deduplication, type casting, validation, and dataset export.
* Exploratory data analysis (EDA), visualizations, charts, heatmaps, business insights, strategic conclusions, and final report writing are handled by downstream team members in subsequent project stages.

## 1. Dataset Overview

### 1.1 Environment Setup
We import core data engineering libraries: `pandas` for tabular manipulation, `numpy` for vectorized numerical operations, and `sys` to audit runtime environment specifications.

In [1]:
import sys
import numpy as np
import pandas as pd

print(f"Python Runtime Version : {sys.version.split()[0]}")
print(f"Pandas Version         : {pd.__version__}")
print(f"NumPy Version          : {np.__version__}")

Python Runtime Version : 3.9.6
Pandas Version         : 2.3.3
NumPy Version          : 2.0.2


### 1.2 Raw Dataset Ingestion and Parsing Strategy
The raw Goodreads dataset (`books.csv`) contains delimited bibliographic records. During initial parsing audits, raw CSV readers encounter parsing exceptions (e.g., C-parser tokenization mismatches) because several records contain unescaped commas within the author or title fields, splitting a single row across 13 fields rather than the expected 12 schema columns.

To achieve robust ingestion in production data engineering pipelines without failing downstream jobs, we ingest the dataset using `on_bad_lines='skip'`, gracefully skipping corrupted records while preserving valid entries.

In [2]:
# Ingest raw dataset with bad-line skipping to bypass malformed records
raw_data_path = 'books.csv'
df_raw = pd.read_csv(raw_data_path, on_bad_lines='skip')

print(f"Raw dataset successfully loaded from '{raw_data_path}'.")
print(f"Initial Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Raw dataset successfully loaded from 'books.csv'.
Initial Shape: 11,123 rows x 12 columns


### 1.3 Column Header Sanitization
Raw CSV headers frequently inherit leading or trailing whitespace artifacts (such as `'  num_pages'`), which introduce schema lookup bugs. We inspect the raw column labels and apply string stripping across all header names.

In [3]:
# Audit and sanitize column headers
print("Original Column Headers:")
print(df_raw.columns.tolist())

# Strip leading and trailing whitespace from column names
df_raw.columns = df_raw.columns.str.strip()

print("\nSanitized Column Headers:")
print(df_raw.columns.tolist())

Original Column Headers:
['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', '  num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']

Sanitized Column Headers:
['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', 'num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']


### 1.4 Raw Sample Inspection
We display the first 5 records of the raw dataset to examine record formatting and preliminary column structures.

In [4]:
# Display the first five rows of the raw dataset
df_raw.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


### 1.5 Structural & Memory Audit (`info`)
We examine the initial schema composition, memory usage, and column storage types assigned during ingestion.

In [5]:
# Structural audit of data types and non-null counts
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11123 entries, 0 to 11122
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11123 non-null  int64  
 1   title               11123 non-null  object 
 2   authors             11123 non-null  object 
 3   average_rating      11123 non-null  float64
 4   isbn                11123 non-null  object 
 5   isbn13              11123 non-null  int64  
 6   language_code       11123 non-null  object 
 7   num_pages           11123 non-null  int64  
 8   ratings_count       11123 non-null  int64  
 9   text_reviews_count  11123 non-null  int64  
 10  publication_date    11123 non-null  object 
 11  publisher           11123 non-null  object 
dtypes: float64(1), int64(5), object(6)
memory usage: 1.0+ MB


## 2. Data Dictionary

The following comprehensive data dictionary documents all 12 columns in the Goodreads dataset, outlining raw storage types, target data types, functional descriptions, domain constraints, and representative examples.

| Column Name | Raw Type | Target Type | Functional Description | Domain Constraints / Validation Rules | Example |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `bookID` | `int64` | `int64` | Unique numerical identifier assigned to each book entry in the Goodreads database. | Positive integer (`> 0`), unique primary key. | `1`, `2` |
| `title` | `object` | `string` | The title of the book, which may include subtitle, series name, or volume number. | Non-empty text string; stripped of leading/trailing spaces. | `Harry Potter and the Half-Blood Prince` |
| `authors` | `object` | `string` | Primary author(s) or creator(s) of the book. Multiple contributors are separated by slashes (`/`). | Non-empty text string; multiple authors delimited by `/`. | `J.K. Rowling/Mary GrandPré` |
| `average_rating` | `float64` | `float64` | Aggregate community mean rating calculated from reader reviews. | Bounded floating point: `0.00 <= average_rating <= 5.00`. | `4.57` |
| `isbn` | `object` | `string` | 10-digit International Standard Book Number identifying the specific book edition. | 10-character string; leading zeros must be preserved. | `0439785960` |
| `isbn13` | `int64` / `object` | `string` | 13-digit International Standard Book Number (EAN barcode format). | 13-character string; leading zeros and digits preserved. | `9780439785969` |
| `language_code` | `object` | `string` | ISO language code denoting the publication language of the edition. | Lowercase standardized string (e.g., `eng`, `spa`, `fre`). | `eng` |
| `num_pages` | `int64` | `int64` | Total page count of the publication. | Positive integer (`num_pages > 0`); 0 indicates missing/audiobook. | `652` |
| `ratings_count` | `int64` | `int64` | Total cumulative count of user ratings submitted by readers. | Non-negative integer (`ratings_count >= 0`). | `2095690` |
| `text_reviews_count` | `int64` | `int64` | Total cumulative count of written qualitative reviews submitted by readers. | Non-negative integer (`text_reviews_count >= 0`); must not exceed `ratings_count`. | `27591` |
| `publication_date` | `object` | `datetime64[ns]` / ISO string | Official release or publication date of the specific edition. | Valid calendar date; formatted to ISO 8601 (`YYYY-MM-DD`). | `2006-09-16` |
| `publisher` | `object` | `string` | Publishing house, imprint, or entity responsible for issuing the edition. | Non-empty text string; cleaned of redundant spaces. | `Scholastic Inc.` |

## 3. Missing Value Analysis

### 3.1 Explicit Null Detection
We assess explicit nulls (`NaN`, `None`, `<NA>`) across each column in the raw dataset.

In [6]:
# Calculate absolute and relative counts of explicit null values
null_summary = pd.DataFrame({
    'Column': df_raw.columns,
    'Null_Count': df_raw.isnull().sum().values,
    'Null_Percentage': ((df_raw.isnull().sum() / len(df_raw)) * 100).round(4).values
})
null_summary.sort_values(by='Null_Count', ascending=False).reset_index(drop=True)

,Column,Null_Count,Null_Percentage
0,bookID,0,0.0
1,title,0,0.0
2,authors,0,0.0
3,average_rating,0,0.0
4,isbn,0,0.0
5,isbn13,0,0.0
6,language_code,0,0.0
7,num_pages,0,0.0
8,ratings_count,0,0.0
9,text_reviews_count,0,0.0


### 3.2 Implicit Missing Value Detection
In addition to standard explicit `NaN` values, real-world data engineering requires identifying **implicit missing values**:
1. **Whitespace / Empty Strings:** Strings containing only whitespace or empty text.
2. **Sentinel Numerical Values:** Books with `num_pages == 0` (often audiobooks or unrecorded pages).
3. **Invalid Calendar Dates:** Dates that fail standard calendar validation rules (e.g., non-existent calendar days like November 31 or June 31).

In [7]:
# Check for empty or whitespace-only text values
text_cols = ['title', 'authors', 'isbn', 'isbn13', 'language_code', 'publisher']
empty_counts = {col: (df_raw[col].astype(str).str.strip() == '').sum() for col in text_cols}
print("Empty / Whitespace-Only String Counts:")
for col, cnt in empty_counts.items():
    print(f"  - {col}: {cnt}")

# Check for books with zero page count
zero_pages_count = (df_raw['num_pages'] == 0).sum()
print(f"\nBooks with num_pages == 0 (Implicit Missing Pages): {zero_pages_count} records")

# Check for invalid calendar dates
parsed_temp_dates = pd.to_datetime(df_raw['publication_date'], format='%m/%d/%Y', errors='coerce')
invalid_dates_mask = parsed_temp_dates.isna()
print(f"\nInvalid Calendar Dates: {invalid_dates_mask.sum()} records")

if invalid_dates_mask.sum() > 0:
    print("\nSample Invalid Calendar Date Records:")
    display(df_raw[invalid_dates_mask][['bookID', 'title', 'authors', 'publication_date', 'publisher']])

Empty / Whitespace-Only String Counts:
  - title: 0
  - authors: 0
  - isbn: 0
  - isbn13: 0
  - language_code: 0
  - publisher: 0

Books with num_pages == 0 (Implicit Missing Pages): 76 records

Invalid Calendar Dates: 2 records

Sample Invalid Calendar Date Records:


,bookID,title,authors,publication_date,publisher
8177,31373,In Pursuit of the Proper Sinner (Inspector Lyn...,Elizabeth George,11/31/2000,Bantam Books
11094,45531,Montaillou village occitan de 1294 à 1324,Emmanuel Le Roy Ladurie/Emmanuel Le Roy-Ladurie,6/31/1982,Folio histoire


## 4. Duplicate Analysis

Data engineering audits require evaluating duplicate records across multiple granularities:
1. **Full-Row Duplicates:** Rows identical across every single attribute.
2. **Primary Key Duplicates:** Duplicates on the unique identifier `bookID`.
3. **ISBN Duplicates:** Redundant entries on `isbn` or `isbn13`.
4. **Entity / Edition Duplicates:** Multiple records sharing the exact same `title` and `authors`.

In [8]:
# Audit duplicates across granularities
exact_row_dups = df_raw.duplicated().sum()
bookid_dups = df_raw['bookID'].duplicated().sum()
isbn_dups = df_raw['isbn'].duplicated().sum()
isbn13_dups = df_raw['isbn13'].duplicated().sum()
title_author_dups = df_raw.duplicated(subset=['title', 'authors'], keep=False).sum()

print("Duplicate Audit Results:")
print(f"  - Exact Full-Row Duplicates       : {exact_row_dups}")
print(f"  - Primary Key (bookID) Duplicates  : {bookid_dups}")
print(f"  - ISBN-10 Duplicates              : {isbn_dups}")
print(f"  - ISBN-13 Duplicates              : {isbn13_dups}")
print(f"  - Duplicate (title, authors) Rows  : {title_author_dups} (affecting multiple editions/formats)")

Duplicate Audit Results:
  - Exact Full-Row Duplicates       : 0
  - Primary Key (bookID) Duplicates  : 0
  - ISBN-10 Duplicates              : 0
  - ISBN-13 Duplicates              : 0
  - Duplicate (title, authors) Rows  : 574 (affecting multiple editions/formats)


### 4.2 Inspection of Duplicate Edition Records
We inspect a sample of records sharing identical `title` and `authors` to verify whether they represent redundant catalog entries or distinct published editions (e.g., paperback vs. audio cassette vs. collector reprint).

In [9]:
# Display sample records with identical title and authors
sample_edition_dups = df_raw[df_raw.duplicated(subset=['title', 'authors'], keep=False)].sort_values(by=['title', 'authors'])
sample_edition_dups[['bookID', 'title', 'authors', 'isbn', 'num_pages', 'ratings_count', 'publisher', 'publication_date']].head(8)

,bookID,title,authors,isbn,num_pages,ratings_count,publisher,publication_date
1574,5415,'Salem's Lot,Stephen King,0965772411,405,1039,Doubleday,10/17/1975
1576,5419,'Salem's Lot,Stephen King,0451092317,427,178,Signet,11/13/1979
1577,5420,'Salem's Lot,Stephen King,0340770538,586,25,Hodder & Stoughton Ltd,10/6/2010
5300,19137,'Salem's Lot,Stephen King,0451098277,817,18,Signet,8/1/1976
9249,36303,'Salem's Lot,Stephen King,0451139690,427,186,Signet,8/1/1976
1573,5414,'Salem's Lot,Stephen King/Ron McLarty,0743536967,17,227,Simon & Schuster Audio,1/19/2004
5298,19135,'Salem's Lot,Stephen King/Ron McLarty,0743536959,0,56,Simon & Schuster Audio,1/19/2004
9130,35552,A Hard Day's Write: The Stories Behind Every B...,Steve Turner,0060844094,224,156,It Books,10/18/2005


## 5. Data Cleaning

In this phase, we execute the systematic cleaning workflow based on our engineering audit:
1. **Text Normalization:** Strip leading/trailing whitespaces and reduce multiple consecutive spaces in string columns (`title`, `authors`, `publisher`, `language_code`).
2. **ISBN String Preservation:** Format `isbn` and `isbn13` strictly as string types, preserving leading zeros and stripping any decimal parsing artifacts.
3. **Duplicate Edition Removal:** To prepare an entity-level clean dataset representing distinct literary works, we deduplicate on `['title', 'authors']`, retaining the edition with the highest `ratings_count` (the primary, most-reviewed edition).
4. **Implicit Missing Value Imputation (`num_pages`):** Impute records with `num_pages == 0` using the median page count of valid physical books.
5. **Relational Review Count Validation:** Fix records where `text_reviews_count > ratings_count` by ensuring `ratings_count = max(ratings_count, text_reviews_count)`.

In [10]:
# Initialize working clean dataframe
df_clean = df_raw.copy()

# Step 1: Text sanitization and internal space reduction
text_columns = ['title', 'authors', 'publisher', 'language_code']
for col in text_columns:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# Step 2: Standardize ISBN strings (protect leading zeros and clean artifacts)
df_clean['isbn'] = df_clean['isbn'].astype(str).str.strip()
df_clean['isbn13'] = df_clean['isbn13'].astype(str).str.split('.').str[0].str.strip()

# Step 3: Entity Deduplication by Title and Authors (Keep edition with highest ratings_count)
initial_rows = len(df_clean)
df_clean = df_clean.sort_values(by='ratings_count', ascending=False)
df_clean = df_clean.drop_duplicates(subset=['title', 'authors'], keep='first')
df_clean = df_clean.sort_values(by='bookID').reset_index(drop=True)
deduped_rows = len(df_clean)
print(f"Deduplication: Removed {initial_rows - deduped_rows:,} duplicate editions. Retained {deduped_rows:,} unique works.")

# Step 4: Impute zero-page records with median of valid publications
valid_pages_median = int(df_clean[df_clean['num_pages'] > 0]['num_pages'].median())
zero_pages_total = (df_clean['num_pages'] == 0).sum()
df_clean.loc[df_clean['num_pages'] == 0, 'num_pages'] = valid_pages_median
print(f"Page Imputation: Replaced {zero_pages_total} zero-page records with median ({valid_pages_median} pages).")

# Step 5: Relational review count validation
anom_reviews_mask = df_clean['text_reviews_count'] > df_clean['ratings_count']
print(f"Review Anomalies: Fixed {anom_reviews_mask.sum()} records where text_reviews_count exceeded ratings_count.")
df_clean.loc[anom_reviews_mask, 'ratings_count'] = df_clean.loc[anom_reviews_mask, 'text_reviews_count']

Deduplication: Removed 320 duplicate editions. Retained 10,803 unique works.
Page Imputation: Replaced 70 zero-page records with median (299 pages).
Review Anomalies: Fixed 2 records where text_reviews_count exceeded ratings_count.


## 6. Data Type Conversion

### 6.1 Publication Date Standardization
During our missing value analysis, we detected two impossible calendar dates in the dataset:
* `11/31/2000` (November has only 30 days) -> Adjusted to valid month-end `11/30/2000`.
* `6/31/1982` (June has only 30 days) -> Adjusted to valid month-end `6/30/1982`.

We convert `publication_date` into proper datetime objects (`datetime64[ns]`), validate the range, and format it to the ISO 8601 standard (`YYYY-MM-DD`) for storage in the clean CSV.

In [11]:
# Correct known calendar typographical errors
df_clean.loc[df_clean['publication_date'] == '11/31/2000', 'publication_date'] = '11/30/2000'
df_clean.loc[df_clean['publication_date'] == '6/31/1982', 'publication_date'] = '6/30/1982'

# Convert to datetime64[ns]
df_clean['publication_date'] = pd.to_datetime(df_clean['publication_date'], format='%m/%d/%Y', errors='raise')

# Format into standardized ISO 8601 date string (YYYY-MM-DD)
df_clean['publication_date'] = df_clean['publication_date'].dt.strftime('%Y-%m-%d')

print("Publication dates successfully standardized to ISO 8601 (YYYY-MM-DD).")
print(f"Sample formatted dates:\n{df_clean['publication_date'].head()}")

Publication dates successfully standardized to ISO 8601 (YYYY-MM-DD).
Sample formatted dates:
0    2006-09-16
1    2004-09-01
2    2003-11-01
3    2004-05-01
4    2004-09-13
Name: publication_date, dtype: object


### 6.2 Numeric and Schema Casting
We enforce strict schema typing across all columns:
* `bookID`: `int64`
* `average_rating`: `float64` (rounded to 2 decimal places)
* `num_pages`: `int64`
* `ratings_count`: `int64`
* `text_reviews_count`: `int64`
* Text/Identifier columns: `object` (string representation)

In [12]:
# Standardize numeric columns
df_clean['bookID'] = df_clean['bookID'].astype('int64')
df_clean['average_rating'] = pd.to_numeric(df_clean['average_rating'], errors='raise').astype('float64').round(2)
df_clean['num_pages'] = df_clean['num_pages'].astype('int64')
df_clean['ratings_count'] = df_clean['ratings_count'].astype('int64')
df_clean['text_reviews_count'] = df_clean['text_reviews_count'].astype('int64')

# Audit converted schema
print("Final Cleaned Schema and Data Types:")
print(df_clean.dtypes)

Final Cleaned Schema and Data Types:
bookID                  int64
title                  object
authors                object
average_rating        float64
isbn                   object
isbn13                 object
language_code          object
num_pages               int64
ratings_count           int64
text_reviews_count      int64
publication_date       object
publisher              object
dtype: object
